# Week-1 defensive analyses (2026-10-27)

A2 = is the module signal present in the matched bulk input?
C2 = are the lost modules in the low-expression regime? where do their transcripts sit?
A3 = power-corrected real-vs-real ceiling
D1 = patient-bootstrap CIs; D2 = permutation null and sepsis-vs-ICU hard negative

Inputs: `bulk_public/GSE279448` (sepsis, 207 bulk samples), `bulk_public/zenodo6497251_imperial_eskd`
(COVID, 336 bulk samples), the pseudobulk npz from `ilisi_floor_20261021/data/pb`, and the
60k balanced cell sets from `ilisi_floor_20261021/data/big`. No model change, no new generation.

## Bulk matrices and Ensembl↔symbol mapping

In [ ]:
import numpy as np, pandas as pd, mygene, os, pickle
PB="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/pb/"
BK="/mnt/user-data/uploads/integration_report_20261012/disease_deg_rare_20261027/bulk_cache/"
ZN="/mnt/user-data/uploads/bulk_public/zenodo6497251_imperial_eskd/"
genes=np.load(PB+"real_sepsis.npz",allow_pickle=True)["genes"].astype(str)
mg=mygene.MyGeneInfo()
r=mg.querymany(list(genes),scopes="symbol",fields="ensembl.gene",species="human",verbose=False)
m={}
for x in r:
    if "ensembl" not in x: continue
    e=x["ensembl"]; ids=[e["gene"]] if isinstance(e,dict) else [y["gene"] for y in e]
    m.setdefault(x["query"],[]).extend(ids)
print("mapped symbols:",len(m),"/",len(genes))
pickle.dump(m,open("out2/sym2ens.pkl","wb"))

# ---- sepsis bulk ----
S=pd.read_csv(BK+"sepsis_bulk_counts.csv.gz",index_col=0)
S.index=[i.split(".")[0] for i in S.index]
print("sepsis bulk",S.shape)
ro=pd.read_csv(PB+"real_sepsis_obs.csv"); fo=pd.read_csv(PB+"floor_sepsis_obs.csv")
gmap={}
for _,x in ro.iterrows():
    gmap[x["sample"]]=("case" if x.group=="sepsis" else "ctrl" if x.group=="healthy control" else "icu" if x.group=="ICU control" else None, x.age_group)
for _,x in fo.iterrows():
    g=x.group
    gmap.setdefault(x["sample"],("case" if g=="adult_sepsis" else "ctrl" if g=="adult_HC" else "icu" if "ICU" in g else None,
                                "paediatric" if g.startswith("ped") else "adult"))
cov=[s for s in S.columns if s in gmap]
print("bulk samples with group:",len(cov),"/",S.shape[1])
meta=pd.DataFrame([(s,)+gmap[s] for s in cov],columns=["sample","cond","age"]).set_index("sample")
print(meta.groupby(["age","cond"]).size())
S[cov].to_pickle("out2/sepsis_bulk.pkl"); meta.to_csv("out2/sepsis_bulk_meta.csv")

# ---- covid bulk ----
C=pd.read_csv(ZN+"htseq_counts.csv",index_col=0)
C.index=[i.split(".")[0] for i in C.index]
w=pd.concat([pd.read_csv(ZN+"w1_metadata.csv"),pd.read_csv(ZN+"w2_metadata.csv")],ignore_index=True)
w=w.drop_duplicates("sample_id").set_index("sample_id")
cc=[s for s in C.columns if s in w.index and w.loc[s,"case_control"] in ("POSITIVE","NEGATIVE")]
cm=pd.DataFrame({"cond":["case" if w.loc[s,"case_control"]=="POSITIVE" else "ctrl" for s in cc],
                 "patient":[w.loc[s,"individual_id"] for s in cc]},index=cc)
print("covid bulk",C.shape,"used",len(cc)); print(cm.cond.value_counts())
C[cc].to_pickle("out2/covid_bulk.pkl"); cm.to_csv("out2/covid_bulk_meta.csv")


## A2 — module signal in the bulk vs in floor

In [ ]:
import numpy as np, pandas as pd, pickle
from scipy import stats
from deg_lib import bh
m=pickle.load(open("out2/sym2ens.pkl","rb"))
allde=pickle.load(open("out/allde.pkl","rb"))

def bulk_deg(X,meta,unit=None):
    tot=X.sum(0); L=np.log2(X/tot*1e6+1)
    L=L.T; L["_c"]=meta.loc[L.index,"cond"].values
    if unit is not None:
        L["_u"]=meta.loc[L.index,unit].values; L=L.groupby(["_u","_c"]).mean().reset_index()
    a=L[L._c=="case"].drop(columns=[c for c in L.columns if c.startswith("_")]).to_numpy()
    b=L[L._c=="ctrl"].drop(columns=[c for c in L.columns if c.startswith("_")]).to_numpy()
    gl=[c for c in L.columns if not c.startswith("_")]
    t,p=stats.ttest_ind(a,b,equal_var=False)
    return pd.DataFrame(dict(ensembl=gl,bulk_logFC=a.mean(0)-b.mean(0),bulk_p=p,bulk_q=bh(p),
                             bulk_mean=(a.mean(0)+b.mean(0))/2,n_case=len(a),n_ctrl=len(b)))

S=pd.read_pickle("out2/sepsis_bulk.pkl"); sm=pd.read_csv("out2/sepsis_bulk_meta.csv",index_col=0)
sa=sm[(sm.age=="adult")&(sm.cond.isin(["case","ctrl"]))]
BS=bulk_deg(S[sa.index],sa)
C=pd.read_pickle("out2/covid_bulk.pkl"); cm=pd.read_csv("out2/covid_bulk_meta.csv",index_col=0)
BC=bulk_deg(C,cm,unit="patient")
print("sepsis bulk n:",BS.n_case[0],"/",BS.n_ctrl[0],"  covid bulk patients:",BC.n_case[0],"/",BC.n_ctrl[0])

# symbol-level bulk table (sum over mapped ensembl ids -> take the most expressed)
def to_symbol(B):
    rows=[]
    bi=B.drop_duplicates("ensembl").set_index("ensembl")
    for s,ids in m.items():
        hit=[i for i in dict.fromkeys(ids) if i in bi.index]
        if not hit: continue
        sub=bi.loc[hit]
        r=sub.iloc[int(np.argmax(sub.bulk_mean.to_numpy()))]
        rows.append((s,float(r.bulk_logFC),float(r.bulk_q),float(r.bulk_mean)))
    return pd.DataFrame(rows,columns=["gene","bulk_logFC","bulk_q","bulk_mean"]).set_index("gene")
bs=to_symbol(BS); bc=to_symbol(BC)
bs["bulk_rank"]=stats.rankdata(-bs.bulk_mean.to_numpy())
bc["bulk_rank"]=stats.rankdata(-bc.bulk_mean.to_numpy())
bs.to_csv("out2/sepsis_bulk_deg.csv"); bc.to_csv("out2/covid_bulk_deg.csv")
print("symbols with bulk values:",len(bs),len(bc))

MOD={"C1Q / complement":("sepsis","Monocyte",["C1QA","C1QB","C1QC","RNASE1","CLU"]),
     "plasma-cell secretory":("sepsis","B cell",["JCHAIN","MZB1","IGLL5","DERL3","TNFRSF17","TXNDC5"]),
     "pDC / cDC identity":("sepsis","DC",["FCER1A","LILRA4","GZMB","PTGDS","CLIC3","CD1C"]),
     "sepsis myeloid activation":("sepsis","Monocyte",["IL1R2","MCEMP1","AQP9","TNFAIP6","ACSL1"]),
     "MHC-II":("sepsis","Monocyte",["HLA-DRA","HLA-DRB1","HLA-DPA1","CD74"]),
     "ISG (COVID)":("covid","Monocyte",["ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1"]),
     "IFI27 / SIGLEC1":("covid","Monocyte",["IFI27","SIGLEC1"])}
rows=[]
for name,(coh,ct,gs) in MOD.items():
    B=bs if coh=="sepsis" else bc
    R=allde[(coh,ct)]["real"].set_index("gene"); F=allde[(coh,ct)]["floor"].set_index("gene")
    for g in gs:
        if g not in B.index or g not in R.index: 
            rows.append(dict(module=name,cohort=coh,celltype=ct,gene=g,note="missing")); continue
        rows.append(dict(module=name,cohort=coh,celltype=ct,gene=g,
            bulk_logFC=B.loc[g,"bulk_logFC"],bulk_q=B.loc[g,"bulk_q"],
            bulk_rank=int(B.loc[g,"bulk_rank"]),
            real_logFC=R.loc[g,"logFC"],real_q=R.loc[g,"q"],floor_logFC=F.loc[g,"logFC"],floor_q=F.loc[g,"q"]))
M=pd.DataFrame(rows); M["bulk_sig"]=M.bulk_q<0.05; M["bulk_same_sign"]=np.sign(M.bulk_logFC)==np.sign(M.real_logFC)
M.to_csv("out2/a2_module_bulk.csv",index=False)
pd.set_option("display.width",250)
print("\n=== per-gene: bulk vs real(sc) vs floor ===")
print(M[["module","gene","bulk_logFC","bulk_q","bulk_sig","real_logFC","floor_logFC"]].round(3).to_string(index=False))
print("\n=== per-module summary ===")
g=M.dropna(subset=["bulk_logFC"]).groupby("module")
sumtab=pd.DataFrame({"n_genes":g.size(),
  "bulk_sig_n":g.bulk_sig.sum(),"bulk_median_logFC":g.bulk_logFC.median(),
  "bulk_sign_match":g.bulk_same_sign.mean(),
  "real_median_logFC":g.real_logFC.median(),"floor_median_logFC":g.floor_logFC.median(),
  "floor/real":g.apply(lambda d:(d.floor_logFC/d.real_logFC).median()),
  "bulk/real":g.apply(lambda d:(d.bulk_logFC/d.real_logFC).median())})
print(sumtab.round(3).to_string())

# background: what fraction of real cell-type DEGs are bulk-detectable?
bg=[]
for (coh,ct),t in allde.items():
    B=bs if coh=="sepsis" else bc
    R=t["real"].set_index("gene"); d=R[R.deg]
    common=[x for x in d.index if x in B.index]
    if len(common)<10: continue
    bb=B.loc[common]
    bg.append(dict(cohort=coh,celltype=ct,n_realDEG=len(common),
                   bulk_sig_frac=float((bb.bulk_q<0.05).mean()),
                   bulk_sign_frac=float((np.sign(bb.bulk_logFC)==np.sign(d.loc[common,"logFC"])).mean()),
                   bulk_logFC_r=float(np.corrcoef(bb.bulk_logFC,d.loc[common,"logFC"])[0,1])))
BG=pd.DataFrame(bg); BG.to_csv("out2/a2_bulk_background.csv",index=False)
print("\n=== background: real cell-type DEGs recovered in bulk ===")
print(BG.round(3).to_string(index=False))


## C2 — expression rank, detection, and UMI concentration

In [ ]:
import numpy as np, pandas as pd, pickle, anndata as ad, scipy.sparse as sp
from scipy import stats
IN="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/big/"
MODG={"C1Q / complement":("sepsis","Monocyte",["C1QA","C1QB","C1QC","RNASE1","CLU"]),
 "plasma-cell secretory":("sepsis","B cell",["JCHAIN","MZB1","IGLL5","DERL3","TNFRSF17","TXNDC5"]),
 "pDC / cDC identity":("sepsis","DC",["FCER1A","LILRA4","GZMB","PTGDS","CLIC3","CD1C"]),
 "sepsis myeloid activation":("sepsis","Monocyte",["IL1R2","MCEMP1","AQP9","TNFAIP6","ACSL1"]),
 "ISG (COVID)":("covid","Monocyte",["ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1"])}
TR="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/results/big/"
rows=[]; conc=[]
for coh in ["sepsis","covid"]:
    R=ad.read_h5ad(f"{IN}{coh}_ref.h5ad"); F=ad.read_h5ad(f"{IN}{coh}_floor.h5ad")
    U=ad.read_h5ad(f"{IN}{coh}_realU.h5ad")
    rng=np.random.default_rng(20261027)
    Xb=sp.csr_matrix(U.X).copy().astype(np.int64)
    tot=np.asarray(Xb.sum(1)).ravel(); Fx=ad.read_h5ad(f"{IN}{coh}_floor.h5ad")
    gt=np.asarray(sp.csr_matrix(Fx.X).sum(1)).ravel(); labF=Fx.obs.six_type.astype(str).to_numpy()
    pool={c:gt[labF==c] for c in np.unique(labF)}
    tgt=np.array([rng.choice(pool.get(c,gt)) for c in U.obs.six_type.astype(str)],float)
    kp=np.minimum(1.0,tgt/np.maximum(tot,1))
    Xb.data=rng.binomial(Xb.data,np.repeat(kp,np.diff(Xb.indptr))); Xb.eliminate_zeros()
    T1=ad.AnnData(Xb.astype(np.float32),obs=U.obs.copy(),var=U.var.copy())
    genes=np.array(R.var_names); gi={g:i for i,g in enumerate(genes)}
    sixR=R.obs.six_type.astype(str).to_numpy(); sixF=F.obs.six_type.astype(str).to_numpy(); sixT=T1.obs.six_type.astype(str).to_numpy()
    XR=sp.csr_matrix(R.X); XF=sp.csr_matrix(F.X); XT=sp.csr_matrix(T1.X)
    for name,(c,ct,gs) in MODG.items():
        if c!=coh: continue
        mR=sixR==ct; mF=sixF==ct; mT=sixT==ct
        # expression rank within that cell type in real A (CP10k mean)
        sub=XR[mR]; tot=np.asarray(sub.sum(1)).ravel(); tot[tot==0]=1
        cp=np.asarray(sp.diags(1e4/tot).dot(sub).mean(0)).ravel()
        rank=stats.rankdata(-cp)
        for g in gs:
            i=gi.get(g)
            if i is None: continue
            dR=float((XR[mR][:,i]>0).mean()); dT=float((XT[mT][:,i]>0).mean()); dF=float((XF[mF][:,i]>0).mean())
            rows.append(dict(module=name,cohort=coh,celltype=ct,gene=g,
                             rank_in_type=int(rank[i]),rank_pct=float(rank[i]/len(genes)),
                             mean_CP10k_realA=float(cp[i]),
                             det_realA=dR,det_realT1=dT,det_floor=dF,
                             det_ratio_floor_over_T1=float(dF/dT) if dT>0 else np.nan))
    # ---- concentration: where do module UMIs sit inside the parent lineage? ----
    if coh=="sepsis":
        for name,ct,gs in [("plasma-cell secretory","B cell",["JCHAIN","MZB1","IGLL5","DERL3","TXNDC5"]),
                           ("C1Q / complement","Monocyte",["C1QA","C1QB","C1QC"])]:
            ii=[gi[g] for g in gs if g in gi]
            for lab,X,mk in [("real A",XR,sixR==ct),("real T1 (depth-matched)",XT,sixT==ct),("floor",XF,sixF==ct)]:
                sub=X[mk]; tot=np.asarray(sub.sum(1)).ravel(); tot[tot==0]=1
                score=np.asarray(sp.diags(1e4/tot).dot(sub)[:,ii].sum(1)).ravel()
                o=np.sort(score)[::-1]; s=o.sum()
                conc.append(dict(module=name,celltype=ct,source=lab,n_cells=int(mk.sum()),
                                 mean_score=float(score.mean()),
                                 pct_cells_pos=float((score>0).mean()*100),
                                 share_top1pct=float(o[:max(1,len(o)//100)].sum()/s*100) if s>0 else np.nan,
                                 share_top5pct=float(o[:max(1,len(o)//20)].sum()/s*100) if s>0 else np.nan,
                                 gini=float(1-2*np.trapezoid(np.cumsum(np.sort(score))/max(score.sum(),1e-9),dx=1/len(score)))))
C=pd.DataFrame(rows); C.to_csv("out2/c2_module_rank.csv",index=False)
K=pd.DataFrame(conc); K.to_csv("out2/c2_concentration.csv",index=False)
pd.set_option("display.width",250)
print("=== module genes: expression rank inside their own cell type (real A) and detection ===")
print(C[["module","gene","rank_in_type","mean_CP10k_realA","det_realA","det_realT1","det_floor","det_ratio_floor_over_T1"]].round(3).to_string(index=False))
print("\n=== rank summary (panel has 5,961 / 5,927 measured genes) ===")
print(C.groupby("module").agg(median_rank=("rank_in_type","median"),worst_rank=("rank_in_type","max"),
                              median_det_realA=("det_realA","median"),median_det_floor=("det_floor","median"),
                              median_det_ratio=("det_ratio_floor_over_T1","median")).round(3).to_string())
print("\n=== where module UMIs sit inside the lineage ===")
print(K.round(2).to_string(index=False))


## C2b — positive-cell fraction vs transcript load (floor and prev K4000)

In [ ]:
import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp
IN="/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/data/big/"
rng=np.random.default_rng(20261027)
SET=[("C1Q / complement","sepsis","Monocyte",["C1QA","C1QB","C1QC"]),
     ("plasma-cell secretory","sepsis","B cell",["JCHAIN","MZB1","IGLL5","DERL3","TXNDC5"]),
     ("pDC / cDC identity","sepsis","DC",["FCER1A","LILRA4","GZMB","PTGDS","CLIC3","CD1C"]),
     ("sepsis myeloid activation","sepsis","Monocyte",["IL1R2","MCEMP1","AQP9","TNFAIP6","ACSL1"]),
     ("ISG (COVID)","covid","Monocyte",["ISG15","IFI6","IFI44L","IFIT1","IFIT3","MX1"])]
rows=[]
for coh in ["sepsis","covid"]:
    R=ad.read_h5ad(f"{IN}{coh}_ref.h5ad"); U=ad.read_h5ad(f"{IN}{coh}_realU.h5ad"); F=ad.read_h5ad(f"{IN}{coh}_floor.h5ad")
    P=ad.read_h5ad(f"{IN}{coh}_prev4k.h5ad")
    g=np.array(R.var_names); gi={x:i for i,x in enumerate(g)}
    XF=sp.csr_matrix(F.X); labF=F.obs.six_type.astype(str).to_numpy()
    Xb=sp.csr_matrix(U.X).copy().astype(np.int64); tot=np.asarray(Xb.sum(1)).ravel()
    gt=np.asarray(XF.sum(1)).ravel(); pool={c:gt[labF==c] for c in np.unique(labF)}
    tgt=np.array([rng.choice(pool.get(c,gt)) for c in U.obs.six_type.astype(str)],float)
    kp=np.minimum(1.0,tgt/np.maximum(tot,1)); Xb.data=rng.binomial(Xb.data,np.repeat(kp,np.diff(Xb.indptr))); Xb.eliminate_zeros()
    SRC={"real A":(sp.csr_matrix(R.X),R.obs.six_type.astype(str).to_numpy()),
         "real (depth-matched)":(sp.csr_matrix(Xb.astype(np.float32)),U.obs.six_type.astype(str).to_numpy()),
         "floor":(XF,labF),
         "prev K4000 (Poisson)":(sp.csr_matrix(P.X),P.obs.six_type.astype(str).to_numpy())}
    for name,c,ct,gs in SET:
        if c!=coh: continue
        ii=[gi[x] for x in gs if x in gi]
        for lab,(X,six) in SRC.items():
            sub=X[six==ct]; t=np.asarray(sub.sum(1)).ravel(); t[t==0]=1
            cp=sp.diags(1e4/t).dot(sub)[:,ii]
            raw=sub[:,ii]
            pos=np.asarray((raw>0).sum(1)).ravel()>0
            sc=np.asarray(cp.sum(1)).ravel()
            umi=np.asarray(raw.sum(1)).ravel()
            rows.append(dict(module=name,celltype=ct,source=lab,n_cells=int((six==ct).sum()),
                pct_pos=100*pos.mean(),
                mean_CP10k_all=float(sc.mean()),
                mean_CP10k_in_pos=float(sc[pos].mean()) if pos.any() else np.nan,
                mean_UMI_in_pos=float(umi[pos].mean()) if pos.any() else np.nan))
D=pd.DataFrame(rows); D.to_csv("out2/c2b_module_load.csv",index=False)
pd.set_option("display.width",230)
print(D.round(2).to_string(index=False))
print("\n=== floor / depth-matched-real ratios ===")
p=D.pivot_table(index=["module"],columns="source",values=["pct_pos","mean_CP10k_in_pos"])
out=pd.DataFrame({"pos% floor/real":p[("pct_pos","floor")]/p[("pct_pos","real (depth-matched)")],
                  "load floor/real":p[("mean_CP10k_in_pos","floor")]/p[("mean_CP10k_in_pos","real (depth-matched)")],
                  "pos% prev/real":p[("pct_pos","prev K4000 (Poisson)")]/p[("pct_pos","real (depth-matched)")],
                  "load prev/real":p[("mean_CP10k_in_pos","prev K4000 (Poisson)")]/p[("mean_CP10k_in_pos","real (depth-matched)")]})
print(out.round(3).to_string())


## A3 / D1 / D2 — power-corrected ceiling, bootstrap CIs, negative controls

In [ ]:
import numpy as np, pandas as pd, pickle
from scipy import stats
from deg_lib import *
rng=np.random.default_rng(20261027)
CT=["T cell","B cell","NK cell","Monocyte","DC"]
SRC={}
for coh in ["sepsis","covid"]:
    for s in ["real","floor"]:
        d=load(f"{s}_{coh}"); SRC[(coh,s)]=(d,design(f"{s}_{coh}",d))
genes={coh:load(f"real_{coh}")["genes"] for coh in ["sepsis","covid"]}
meas={}
for coh in ["sepsis","covid"]:
    r=load(f"real_{coh}"); f=load(f"floor_{coh}")
    meas[coh]=set(r["genes"][(r["S"].sum(0)>0)&(f["S"].sum(0)>0)])

def mat(coh,src,ct):
    d,des=SRC[(coh,src)]; return cpm_matrix(d,des,ct)
def tab(g,coh,units_case=None,units_ctrl=None):
    if units_case is not None:
        g=g[((g._c=="case")&g._u.isin(units_case))|((g._c=="ctrl")&g._u.isin(units_ctrl))]
    t=deg(g,list(genes[coh]))
    return None if t is None else t[t.gene.isin(meas[coh])].reset_index(drop=True)
def cmp(A,B):
    B=B.set_index("gene").loc[A.gene]
    r=float(np.corrcoef(A.logFC,B.logFC)[0,1])
    rec=float((A.deg.values&B.deg.values).sum()/max(A.deg.sum(),1)) if A.deg.sum() else np.nan
    pre=float((A.deg.values&B.deg.values).sum()/max(B.deg.sum(),1)) if B.deg.sum() else np.nan
    return r,rec,pre

# ================= A3: power-matched ceiling =================
rows=[]
for coh in ["sepsis","covid"]:
    for ct in CT:
        gR=mat(coh,"real",ct); gF=mat(coh,"floor",ct)
        if len(gR)==0 or len(gF)==0: continue
        cR=gR[gR._c=="case"]._u.unique(); oR=gR[gR._c=="ctrl"]._u.unique()
        full=tab(gR,coh); fl=tab(gF,coh)
        if full is None or fl is None: continue
        r_full,rec_full,pre_full=cmp(full,fl)
        acc=[]
        for _ in range(30):
            ca=rng.permutation(cR); h=len(ca)//2
            # V1: split case arm only, controls shared
            A=tab(gR,coh,ca[:h],oR); B=tab(gR,coh,ca[h:],oR)
            if A is None or B is None: continue
            v1=cmp(A,B)
            # V2: floor subsampled to the same unit counts as the halves
            cF=gF[gF._c=="case"]._u.unique(); oF=gF[gF._c=="ctrl"]._u.unique()
            fs=tab(gF,coh,rng.choice(cF,min(h,len(cF)),replace=False),rng.choice(oF,min(len(oR),len(oF)),replace=False))
            v2=cmp(A,fs) if fs is not None else (np.nan,)*3
            acc.append(v1+v2)
        if not acc: continue
        a=np.array(acc,float)
        rows.append(dict(cohort=coh,celltype=ct,n_case=len(cR),n_ctrl=len(oR),
            floor_r_full=r_full,floor_recall_full=rec_full,floor_prec_full=pre_full,
            ceil_case_split_r=np.nanmean(a[:,0]),ceil_case_split_r_sd=np.nanstd(a[:,0]),
            ceil_recall=np.nanmean(a[:,1]),ceil_prec=np.nanmean(a[:,2]),
            floor_r_matchedN=np.nanmean(a[:,3]),floor_recall_matchedN=np.nanmean(a[:,4]),floor_prec_matchedN=np.nanmean(a[:,5])))
A3=pd.DataFrame(rows)
A3["pct_of_ceiling_r"]=100*A3.floor_r_matchedN/A3.ceil_case_split_r
A3.to_csv("out2/a3_power_matched_ceiling.csv",index=False)
pd.set_option("display.width",260)
print("=== A3: control-arm-shared ceiling, floor at matched n ===")
print(A3[["cohort","celltype","n_case","n_ctrl","ceil_case_split_r","ceil_case_split_r_sd","floor_r_matchedN","pct_of_ceiling_r","ceil_recall","floor_recall_matchedN"]].round(3).to_string(index=False))

# ================= D1: patient bootstrap CI =================
rows=[]
for coh in ["sepsis","covid"]:
    for ct in CT:
        gR=mat(coh,"real",ct); gF=mat(coh,"floor",ct)
        if len(gR)==0 or len(gF)==0: continue
        cR=gR[gR._c=="case"]._u.unique(); oR=gR[gR._c=="ctrl"]._u.unique()
        cF=gF[gF._c=="case"]._u.unique(); oF=gF[gF._c=="ctrl"]._u.unique()
        boot=[]
        for _ in range(200):
            A=tab(gR,coh,rng.choice(cR,len(cR),True),rng.choice(oR,len(oR),True))
            B=tab(gF,coh,rng.choice(cF,len(cF),True),rng.choice(oF,len(oF),True))
            if A is None or B is None: continue
            boot.append(cmp(A,B))
        if not boot: continue
        b=np.array(boot,float)
        q=lambda i:(np.nanpercentile(b[:,i],2.5),np.nanpercentile(b[:,i],97.5))
        full=tab(gR,coh); fl=tab(gF,coh); r,rec,pre=cmp(full,fl)
        rows.append(dict(cohort=coh,celltype=ct,logFC_r=r,r_lo=q(0)[0],r_hi=q(0)[1],
                         recall=rec,rec_lo=q(1)[0],rec_hi=q(1)[1],precision=pre,pre_lo=q(2)[0],pre_hi=q(2)[1]))
D1=pd.DataFrame(rows); D1.to_csv("out2/d1_bootstrap_ci.csv",index=False)
print("\n=== D1: patient bootstrap 95% CI (200 reps) ===")
print(D1.round(3).to_string(index=False))

# ================= D2: negative controls =================
rows=[]
for coh in ["sepsis","covid"]:
    for ct in CT:
        gR=mat(coh,"real",ct); gF=mat(coh,"floor",ct)
        if len(gR)==0 or len(gF)==0: continue
        # permutation null: shuffle condition labels in BOTH, 20 reps
        perm=[]
        for _ in range(20):
            a=gR.copy(); a["_c"]=rng.permutation(a._c.values)
            b=gF.copy(); b["_c"]=rng.permutation(b._c.values)
            A=tab(a,coh); B=tab(b,coh)
            if A is None or B is None: continue
            perm.append(cmp(A,B))
        p=np.array(perm,float) if perm else np.full((1,3),np.nan)
        full=tab(gR,coh); fl=tab(gF,coh); r,rec,pre=cmp(full,fl)
        rows.append(dict(cohort=coh,celltype=ct,observed_r=r,perm_r_mean=np.nanmean(p[:,0]),
                         perm_r_p975=np.nanpercentile(p[:,0],97.5),
                         perm_nDEG_real=np.nanmean([np.nan]),observed_recall=rec,perm_recall=np.nanmean(p[:,1])))
D2=pd.DataFrame(rows).drop(columns=["perm_nDEG_real"])
D2.to_csv("out2/d2_permutation_null.csv",index=False)
print("\n=== D2a: label-permutation null ===")
print(D2.round(3).to_string(index=False))

# D2b: sepsis vs ICU control (known hard negative)
import copy
rows=[]
d=load("real_sepsis"); o=d["obs"]; o=o[o.age_group=="adult"]
dr=o.assign(cond=o.group.map({"sepsis":"case","ICU control":"ctrl"})).dropna(subset=["cond"]).assign(unit=lambda x:x["sample"])[["sample","unit","cond"]]
df=load("floor_sepsis"); of=df["obs"]
dfm=of.assign(cond=of.group.map({"adult_sepsis":"case","adult_ICU_ctrl":"ctrl"})).dropna(subset=["cond"]).assign(unit=lambda x:x["sample"])[["sample","unit","cond"]]
for ct in CT:
    A=deg(cpm_matrix(d,dr,ct),list(genes["sepsis"])); B=deg(cpm_matrix(df,dfm,ct),list(genes["sepsis"]))
    if A is None or B is None: continue
    A=A[A.gene.isin(meas["sepsis"])].reset_index(drop=True); B=B[B.gene.isin(meas["sepsis"])].reset_index(drop=True)
    r,rec,pre=cmp(A,B)
    rows.append(dict(celltype=ct,n_realDEG=int(A.deg.sum()),n_floorDEG=int(B.deg.sum()),logFC_r=r,recall=rec,precision=pre))
D2b=pd.DataFrame(rows); D2b.to_csv("out2/d2b_sepsis_vs_icu.csv",index=False)
print("\n=== D2b: sepsis vs ICU control (hard negative) ===")
print(D2b.round(3).to_string(index=False))


## Figures

In [ ]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.rcParams.update({"font.family":"DejaVu Sans","axes.spines.top":False,"axes.spines.right":False,
 "font.size":9,"axes.labelsize":9,"axes.titlesize":10,"legend.fontsize":8,"xtick.labelsize":8,"ytick.labelsize":8})
BLUE,RED,GREY,INK,GREEN,ORANGE="#2B83BA","#D7191C","#7F7F7F","#1F2933","#1A9641","#E08214"
F="figs/"; save=lambda fig,n:(fig.savefig(F+n+".png",dpi=220,bbox_inches="tight",facecolor="white"),plt.close(fig),print("saved",n))

# --- A2: is the module in the bulk? ---
M=pd.read_csv("out2/a2_module_bulk.csv").dropna(subset=["bulk_logFC"])
g=M.groupby("module")
S=pd.DataFrame({"bulk/real":g.apply(lambda d:(d.bulk_logFC/d.real_logFC).median()),
                "floor/real":g.apply(lambda d:(d.floor_logFC/d.real_logFC).median())})
order=["pDC / cDC identity","C1Q / complement","IFI27 / SIGLEC1","MHC-II","ISG (COVID)","sepsis myeloid activation"]
S=S.loc[[o for o in order if o in S.index]]
fig,ax=plt.subplots(figsize=(8.6,4.0)); x=np.arange(len(S)); w=0.38
ax.bar(x-w/2,S["bulk/real"],w,color="#C9B8D8",edgecolor="#7B3294",lw=1.0,label="bulk input")
ax.bar(x+w/2,S["floor/real"],w,color="#A8CDE3",edgecolor=BLUE,lw=1.0,label="PBISC floor")
ax.axhline(1,ls="--",lw=.9,c=INK); ax.axhline(0.5,ls=":",lw=.9,c=RED)
for i,(b,f) in enumerate(zip(S["bulk/real"],S["floor/real"])):
    ax.text(i-w/2,b+0.03,f"{b:.2f}",ha="center",fontsize=8); ax.text(i+w/2,f+0.03,f"{f:.2f}",ha="center",fontsize=8,color=BLUE)
ax.set_xticks(x); ax.set_xticklabels([s.replace(" / ","/\n") for s in S.index],fontsize=8)
ax.set_ylabel("module disease log2FC / real single-cell"); ax.set_ylim(0,1.6)
ax.set_title("The signal is in the bulk — floor fails to route it to the cell type",loc="left")
fig.legend(frameon=False,loc="lower center",bbox_to_anchor=(0.5,-0.02),ncol=2,columnspacing=2.4)
fig.tight_layout(rect=[0,0.07,1,1]); save(fig,"w1_bulk_vs_floor")

# --- C2b: positivity vs load ---
L=pd.read_csv("out2/c2b_module_load.csv")
p=L.pivot_table(index="module",columns="source",values=["pct_pos","mean_CP10k_in_pos"])
R=pd.DataFrame({"pos%":p[("pct_pos","floor")]/p[("pct_pos","real (depth-matched)")],
                "load":p[("mean_CP10k_in_pos","floor")]/p[("mean_CP10k_in_pos","real (depth-matched)")],
                "pos%_prev":p[("pct_pos","prev K4000 (Poisson)")]/p[("pct_pos","real (depth-matched)")],
                "load_prev":p[("mean_CP10k_in_pos","prev K4000 (Poisson)")]/p[("mean_CP10k_in_pos","real (depth-matched)")]})
R=R.loc[[o for o in ["C1Q / complement","pDC / cDC identity","plasma-cell secretory","sepsis myeloid activation","ISG (COVID)"] if o in R.index]]
fig,ax=plt.subplots(1,2,figsize=(10.6,4.0))
x=np.arange(len(R)); w=0.38
for j,(ttl,c1,c2) in enumerate([("PBISC floor","pos%","load"),("prev K4000 (Poisson read-out)","pos%_prev","load_prev")]):
    ax[j].bar(x-w/2,R[c1],w,color="#D5DBE1",edgecolor=GREY,lw=1.0)
    ax[j].bar(x+w/2,R[c2],w,color="#A8CDE3" if j==0 else "#F2A7A8",edgecolor=BLUE if j==0 else RED,lw=1.0)
    ax[j].axhline(1,ls="--",lw=.9,c=INK)
    for i,(a,b) in enumerate(zip(R[c1],R[c2])):
        ax[j].text(i-w/2,a+0.03,f"{a:.2f}",ha="center",fontsize=7.5)
        ax[j].text(i+w/2,b+0.03,f"{b:.2f}",ha="center",fontsize=7.5)
    ax[j].set_xticks(x); ax[j].set_xticklabels([s.replace(" / ","/\n").replace(" secretory","\nsecretory").replace(" activation","\nactivation") for s in R.index],fontsize=7.5)
    ax[j].set_ylim(0,2.0); ax[j].set_title(ttl,loc="left"); ax[j].set_ylabel("ratio vs depth-matched real")
fig.legend(handles=[plt.Rectangle((0,0),1,1,fc="#D5DBE1",ec=GREY),plt.Rectangle((0,0),1,1,fc="#A8CDE3",ec=BLUE)],
           labels=["% of cells positive for the module","transcript load inside positive cells"],
           frameon=False,loc="lower center",bbox_to_anchor=(0.5,-0.02),ncol=2,columnspacing=2.4)
fig.suptitle("Right number of positive cells, wrong amount inside them",x=0.005,ha="left",fontsize=10.5,fontweight="bold")
fig.tight_layout(rect=[0,0.08,1,0.95]); save(fig,"w1_pos_vs_load")

# --- A3: ceiling bracket ---
A=pd.read_csv("out2/a3_power_matched_ceiling.csv"); C=pd.read_csv("out/real_real_ceiling.csv")
A=A.merge(C[["cohort","celltype","logFC_r"]].rename(columns={"logFC_r":"ceil_bothsplit"}),on=["cohort","celltype"])
fig,ax=plt.subplots(1,2,figsize=(10.6,3.9))
CT=["T cell","B cell","NK cell","Monocyte","DC"]
for j,coh in enumerate(["sepsis","covid"]):
    d=A[A.cohort==coh].set_index("celltype").loc[CT]; x=np.arange(len(CT))
    ax[j].fill_between(x,d.ceil_bothsplit,d.ceil_case_split_r,color=INK,alpha=.12,step="mid")
    ax[j].plot(x,d.ceil_bothsplit,"_",ms=28,color=GREY,mew=2)
    ax[j].plot(x,d.ceil_case_split_r,"_",ms=28,color=INK,mew=2)
    ax[j].bar(x,d.floor_r_full,0.45,color="#A8CDE3",edgecolor=BLUE,lw=1.0)
    for i,t in enumerate(CT):
        lo=100*d.floor_r_full[t]/d.ceil_case_split_r[t]; hi=100*d.floor_r_full[t]/d.ceil_bothsplit[t]
        ax[j].text(i,d.floor_r_full[t]+0.025,f"{lo:.0f}–{min(hi,999):.0f}%",ha="center",fontsize=7.5,color=BLUE)
    ax[j].set_xticks(x); ax[j].set_xticklabels(CT,rotation=20,ha="right"); ax[j].set_ylim(0,1.05)
    ax[j].set_ylabel("disease logFC Pearson r"); ax[j].set_title({"sepsis":"Sepsis","covid":"COVID / ESKD"}[coh],loc="left")
fig.legend(handles=[Line2D([],[],color=GREY,lw=2),Line2D([],[],color=INK,lw=2),plt.Rectangle((0,0),1,1,fc="#A8CDE3",ec=BLUE)],
           labels=["ceiling, both arms split (under-powered → low)","ceiling, controls shared (correlated → high)","PBISC floor"],
           frameon=False,loc="lower center",bbox_to_anchor=(0.5,-0.02),ncol=3,columnspacing=1.8)
fig.suptitle("The ceiling is itself uncertain — report floor as a bracket",x=0.005,ha="left",fontsize=10.5,fontweight="bold")
fig.tight_layout(rect=[0,0.09,1,0.95]); save(fig,"w1_ceiling_bracket")

# --- D1 CI forest ---
D=pd.read_csv("out2/d1_bootstrap_ci.csv")
fig,ax=plt.subplots(figsize=(7.4,4.2)); y=np.arange(len(D))[::-1]
ax.hlines(y,D.r_lo,D.r_hi,color=GREY,lw=2.2)
ax.plot(D.logFC_r,y,"o",color=BLUE,ms=7)
ax.axvline(0,lw=.8,c=INK)
ax.set_yticks(y); ax.set_yticklabels([f"{r.cohort[:3]}  {r.celltype}" for r in D.itertuples()],fontsize=8)
ax.set_xlabel("disease logFC r vs real  (point estimate, 95% patient-bootstrap CI)")
ax.set_xlim(-0.05,1.0); ax.set_title("Uncertainty of the headline metric",loc="left")
fig.tight_layout(); save(fig,"w1_bootstrap_ci")


## Outputs
`out2/a2_module_bulk.csv`, `out2/a2_bulk_background.csv`, `out2/sepsis_bulk_deg.csv`, `out2/covid_bulk_deg.csv`,
`out2/c2_module_rank.csv`, `out2/c2_concentration.csv`, `out2/c2b_module_load.csv`,
`out2/a3_power_matched_ceiling.csv`, `out2/d1_bootstrap_ci.csv`, `out2/d2_permutation_null.csv`, `out2/d2b_sepsis_vs_icu.csv`